<a href="https://colab.research.google.com/github/doyun1119/-/blob/main/Rogue(1980)_%EB%A7%8C%EB%93%A4%EA%B8%B0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 제 1장 게임의 기본 구조

## 1-1 개발 환경 준비

In [1]:
import random
import os
import time

print("Rogue 개발환경을 준비 했습니다")
print("Python 버전:",__import__("sys").version.split()[0])

Rogue 개발환경을 준비 했습니다
Python 버전: 3.13.15


## 1-2 ASCII 던전 화면 만들기

In [2]:
#던전 크기
MAP_WIDTH = 40
MAP_HEIGHT = 20

#던전 타일
WALL = "#"
FLOOR = "."

#플레이어
PLAYER = "@"

#테스트용 던전
dungeon = [list("#"*MAP_WIDTH) for _ in range(MAP_HEIGHT)]

#가운데에 방 만들기
for y in range(2, MAP_HEIGHT - 2):
  for x in range(2, MAP_WIDTH - 2):
    dungeon[y][x] = FLOOR

#플레이어 위치
player_x = MAP_WIDTH // 2
player_y = MAP_HEIGHT // 2

#플레이어 표시
dungeon[player_y][player_x] = PLAYER

def draw_map():
  """현재 던전을 화면에 출력한다"""

  # Colab 화면을 어느정도 깔끔하게 유지
  print("\n" * 2)

  for row in dungeon:
    print("".join(row))


draw_map()




########################################
########################################
##....................................##
##....................................##
##....................................##
##....................................##
##....................................##
##....................................##
##....................................##
##....................................##
##..................@.................##
##....................................##
##....................................##
##....................................##
##....................................##
##....................................##
##....................................##
##....................................##
########################################
########################################


## 1-3 플래이어 이동

In [3]:
def move_player(dx,dy):
  """
  플래이어를 dx,dy만큼 이동시킨다.
  이동할 수 있으면 True,
  이동할수 없으면 False를 반환한다.
  """

  global player_x,player_y

  new_x = player_x + dx
  new_y = player_y + dy

  #던전 범위를 벗어나는지 확인
  if new_x < 0 or new_x >= MAP_WIDTH:
        return False

  if new_y < 0 or new_y >= MAP_HEIGHT:
        return False

  # 벽인지 확인
  if dungeon[new_y][new_x] == WALL:
        return False

  # 기존 위치를 바닥으로 변경
  dungeon[player_y][player_x] = FLOOR

  # 플레이어 위치 변경
  player_x = new_x
  player_y = new_y

  # 새로운 위치에 플레이어 표시
  dungeon[player_y][player_x] = PLAYER

  return True


def process_input(command):
    """
    플레이어의 입력을 처리한다.
    """

    command = command.lower()

    if command == "w":
        return move_player(0, -1)

    elif command == "s":
        return move_player(0, 1)

    elif command == "a":
        return move_player(-1, 0)

    elif command == "d":
        return move_player(1, 0)

    return None

## 1-4 기본 게임 루프

In [4]:
def game_loop():
    """게임 전체를 실행한다."""

    print("=" * 40)
    print("         ROGUE")
    print("   ASCII Dungeon Adventure")
    print("=" * 40)

    print()
    print("W : 위")
    print("A : 왼쪽")
    print("S : 아래")
    print("D : 오른쪽")
    print("Q : 게임 종료")
    print()

    while True:

        # 화면 출력
        draw_map()

        # 현재 플레이어 위치
        print()
        print(f"위치: ({player_x}, {player_y})")

        # 명령 입력
        command = input("\n명령을 입력하세요: ")

        # 게임 종료
        if command.lower() == "q":
            print("\n게임을 종료합니다.")
            break

        # 이동 처리
        result = process_input(command)

        if result is True:
            print("이동했습니다.")

        elif result is False:
            print("벽 때문에 이동할 수 없습니다.")

        else:
            print("알 수 없는 명령입니다.")


# 게임 시작
game_loop()

         ROGUE
   ASCII Dungeon Adventure

W : 위
A : 왼쪽
S : 아래
D : 오른쪽
Q : 게임 종료




########################################
########################################
##....................................##
##....................................##
##....................................##
##....................................##
##....................................##
##....................................##
##....................................##
##....................................##
##..................@.................##
##....................................##
##....................................##
##....................................##
##....................................##
##....................................##
##....................................##
##....................................##
########################################
########################################

위치: (20, 10)

명령을 입력하세요: q

게임을 종료합니다.


# 제 2장 랜덤 던전 생성


## 2-1 방 생성

In [5]:
import random

# 던전 크기
MAP_WIDTH = 60
MAP_HEIGHT = 25

# 타일
WALL = "#"
FLOOR = "."

# 방 생성 설정
MIN_ROOM_WIDTH = 5
MAX_ROOM_WIDTH = 12

MIN_ROOM_HEIGHT = 4
MAX_ROOM_HEIGHT = 8

MAX_ROOMS = 8


class Room:
    """던전의 방을 나타내는 클래스"""

    def __init__(self, x, y, width, height):
        self.x = x
        self.y = y
        self.width = width
        self.height = height

    def center(self):
        """방의 중심 좌표를 반환한다."""
        center_x = self.x + self.width // 2
        center_y = self.y + self.height // 2

        return center_x, center_y

    def overlaps(self, other):
        """다른 방과 겹치는지 확인한다."""

        return (
            self.x <= other.x + other.width
            and self.x + self.width >= other.x
            and self.y <= other.y + other.height
            and self.y + self.height >= other.y
        )


def create_empty_dungeon():
    """벽으로 가득 찬 빈 던전을 만든다."""

    dungeon = []

    for y in range(MAP_HEIGHT):
        row = []

        for x in range(MAP_WIDTH):
            row.append(WALL)

        dungeon.append(row)

    return dungeon


def create_room(dungeon, room):
    """던전에 방을 만든다."""

    for y in range(room.y, room.y + room.height):
        for x in range(room.x, room.x + room.width):
            dungeon[y][x] = FLOOR


def generate_rooms():
    """랜덤한 방들을 생성한다."""

    rooms = []

    for _ in range(MAX_ROOMS):

        width = random.randint(
            MIN_ROOM_WIDTH,
            MAX_ROOM_WIDTH
        )

        height = random.randint(
            MIN_ROOM_HEIGHT,
            MAX_ROOM_HEIGHT
        )

        x = random.randint(
            1,
            MAP_WIDTH - width - 2
        )

        y = random.randint(
            1,
            MAP_HEIGHT - height - 2
        )

        new_room = Room(
            x,
            y,
            width,
            height
        )

        # 기존 방과 겹치는지 확인
        overlaps = False

        for room in rooms:
            if new_room.overlaps(room):
                overlaps = True
                break

        # 겹치지 않는 경우만 추가
        if not overlaps:
            rooms.append(new_room)

    return rooms


# 던전 생성
dungeon = create_empty_dungeon()

# 방 생성
rooms = generate_rooms()

# 던전에 방 표시
for room in rooms:
    create_room(dungeon, room)


# 던전 출력
for row in dungeon:
    print("".join(row))


print()
print("생성된 방:", len(rooms))

############################################################
############################################################
###############################################...........##
###################################.......#####...........##
####.........######################.......#####...........##
####.........######################.......#####...........##
####.........######################.......#####...........##
####.........######################.......#####...........##
####.........######################.......##################
####.........###############################################
####.........###############################################
############################################################
############################################################
#############.........######################################
#############.........######################################
#############.........######################################
#############.........##

## 2-2 통로 연결

In [6]:
def create_horizontal_corridor(dungeon, x1, x2, y):
    """두 지점을 가로 통로로 연결한다."""

    start_x = min(x1, x2)
    end_x = max(x1, x2)

    for x in range(start_x, end_x + 1):
        dungeon[y][x] = FLOOR


def create_vertical_corridor(dungeon, y1, y2, x):
    """두 지점을 세로 통로로 연결한다."""

    start_y = min(y1, y2)
    end_y = max(y1, y2)

    for y in range(start_y, end_y + 1):
        dungeon[y][x] = FLOOR


def connect_rooms(dungeon, room1, room2):
    """두 방을 통로로 연결한다."""

    x1, y1 = room1.center()
    x2, y2 = room2.center()

    # 먼저 가로로 이동한 뒤 세로로 이동
    create_horizontal_corridor(
        dungeon,
        x1,
        x2,
        y1
    )

    create_vertical_corridor(
        dungeon,
        y1,
        y2,
        x2
    )


# 방들을 순서대로 연결
for i in range(1, len(rooms)):
    previous_room = rooms[i - 1]
    current_room = rooms[i]

    connect_rooms(
        dungeon,
        previous_room,
        current_room
    )


# 결과 출력
for row in dungeon:
    print("".join(row))

############################################################
############################################################
###############################################...........##
###################################.......#####...........##
####.........######################.......#####...........##
####.........######################.......##..............##
####......................................##.##...........##
####......................................................##
####.........######################.......##.###############
####.........#########################.#####.###############
####.........#########################.#####.###############
######################################.#####.###############
######################################.#####.###############
#############.........################.#####.###############
#############.........################.#####.###############
#############..........................#####.###############
#############.........##

## 2-3 랜덤 던전 완성

In [7]:
def generate_dungeon():
  """완성된 랜덤 전전을 생성한다."""

  # 빈 던전 생성
  dungeon = create_empty_dungeon()

  # 방 생성
  rooms = generate_rooms()

  # 방을 던전에 추가
  for room in rooms:
    create_room(dungeon, room)

  # 방들을 통로로 연결
  for i in range(1, len(rooms)):
      connect_rooms(
          dungeon,
          rooms[i - 1],
          rooms[i]
      )

  return dungeon, rooms


# 새로운 던전 생성
dungeon, rooms = generate_dungeon()


# 던전 출력
for row in dungeon:
    print("".join(row))


print()
print("방의 개수:", len(rooms))

############################################################
###################################################.....####
###################################################.....####
###################################################.....####
###################################################.....####
#####################################################.######
#########################.........###################.######
#########################.........################......####
#########################.........################......####
#########################...............................####
#########################.........#.##############......####
#########################.........#.##############......####
###################################.########################
##########............#############.########################
##########............#############.########################
##########............#############.########################
##########............##

## 2-4 던전 검증

In [8]:
def get_floor_positions(dungeon):
    """던전에서 모든 바닥 위치를 찾는다."""

    positions = []

    for y in range(MAP_HEIGHT):
        for x in range(MAP_WIDTH):

            if dungeon[y][x] == FLOOR:
                positions.append((x, y))

    return positions


def flood_fill(dungeon, start_x, start_y):
    """시작점에서 이동 가능한 모든 바닥을 찾는다."""

    visited = set()
    stack = [(start_x, start_y)]

    directions = [
        (0, -1),  # 위
        (0, 1),   # 아래
        (-1, 0),  # 왼쪽
        (1, 0)    # 오른쪽
    ]

    while stack:

        x, y = stack.pop()

        # 이미 방문한 위치라면 무시
        if (x, y) in visited:
            continue

        # 범위를 벗어나면 무시
        if x < 0 or x >= MAP_WIDTH:
            continue

        if y < 0 or y >= MAP_HEIGHT:
            continue

        # 벽이면 이동할 수 없음
        if dungeon[y][x] != FLOOR:
            continue

        # 방문 처리
        visited.add((x, y))

        # 주변 위치 추가
        for dx, dy in directions:
            next_x = x + dx
            next_y = y + dy

            stack.append((next_x, next_y))

    return visited


def validate_dungeon(dungeon, rooms):
    """던전이 정상적으로 연결되어 있는지 검사한다."""

    if len(rooms) == 0:
        return False

    # 첫 번째 방의 중심에서 시작
    start_x, start_y = rooms[0].center()

    # 이동 가능한 모든 바닥 탐색
    reachable = flood_fill(
        dungeon,
        start_x,
        start_y
    )

    # 모든 방의 중심에 도달할 수 있는지 확인
    for room in rooms:

        x, y = room.center()

        if (x, y) not in reachable:
            return False

    return True


# 던전 검사
is_valid = validate_dungeon(
    dungeon,
    rooms
)

print()
print("던전 검증 결과:", is_valid)


던전 검증 결과: True


# 제 3장 플래이어 시스템

## 3-1 플래이어 기본 정보

In [9]:
class Player:
    def __init__(self, x, y):
        self.x = x
        self.y = y

        self.max_hp = 20
        self.hp = 20

        self.attack = 5
        self.defense = 2

        self.level = 1
        self.xp = 0
        self.next_xp = 10

        self.alive = True

print("Player 클래스 생성 완료")

Player 클래스 생성 완료


## 3-2 플레이어 생성

In [10]:
player_x, player_y = rooms[0].center()

player = Player(
    player_x,
    player_y
)

print("플레이어 생성 완료")
print("위치:", player.x, player.y)
print("HP:", player.hp, "/", player.max_hp)
print("공격력:", player.attack)
print("방어력:", player.defense)

플레이어 생성 완료
위치: 53 3
HP: 20 / 20
공격력: 5
방어력: 2


## 3-3 플레이어 이동

In [11]:
def move_player(dx, dy):

    new_x = player.x + dx
    new_y = player.y + dy

    # 맵 범위 확인
    if new_x < 0 or new_x >= MAP_WIDTH:
        print("맵 밖으로 이동할 수 없습니다.")
        return

    if new_y < 0 or new_y >= MAP_HEIGHT:
        print("맵 밖으로 이동할 수 없습니다.")
        return

    # 벽 확인
    if dungeon[new_y][new_x] == WALL:
        print("벽에 막혔습니다.")
        return

    # 이동
    player.x = new_x
    player.y = new_y

    print(
        "현재 위치:",
        player.x,
        player.y
    )


def draw_player_map():

    for y in range(MAP_HEIGHT):

        row = ""

        for x in range(MAP_WIDTH):

            if (
                x == player.x
                and y == player.y
            ):
                row += "@"

            else:
                row += dungeon[y][x]

        print(row)


draw_player_map()

############################################################
###################################################.....####
###################################################.....####
###################################################..@..####
###################################################.....####
#####################################################.######
#########################.........###################.######
#########################.........################......####
#########################.........################......####
#########################...............................####
#########################.........#.##############......####
#########################.........#.##############......####
###################################.########################
##########............#############.########################
##########............#############.########################
##########............#############.########################
##########............##

## 3-4 플레이어 상태창

In [12]:
def show_player_status():

    print("==============================")
    print("        PLAYER STATUS")
    print("==============================")

    print(
        f"HP      : {player.hp}/{player.max_hp}"
    )

    print(
        f"ATK     : {player.attack}"
    )

    print(
        f"DEF     : {player.defense}"
    )

    print(
        f"LEVEL   : {player.level}"
    )

    print(
        f"XP      : {player.xp}/{player.next_xp}"
    )

    print(
        f"POSITION: ({player.x}, {player.y})"
    )

    print("==============================")


show_player_status()

        PLAYER STATUS
HP      : 20/20
ATK     : 5
DEF     : 2
LEVEL   : 1
XP      : 0/10
POSITION: (53, 3)


## 3-5 턴 시스템

In [13]:
turn = 0


def player_wait():

    global turn

    turn += 1

    print(
        f"{turn}턴이 진행되었습니다."
    )


def player_move_turn(dx, dy):

    global turn

    old_x = player.x
    old_y = player.y

    move_player(dx, dy)

    # 실제로 이동했을 때만 턴 증가
    if (
        player.x != old_x
        or
        player.y != old_y
    ):
        turn += 1

    print("현재 턴:", turn)


print("턴 시스템 준비 완료")

턴 시스템 준비 완료


## 3-6 경험치 시스템

In [14]:
def gain_experience(amount):

    player.xp += amount

    print(
        f"경험치 +{amount}"
    )

    print(
        f"현재 XP: {player.xp}/{player.next_xp}"
    )


gain_experience(3)

경험치 +3
현재 XP: 3/10


## 3-7 레벨업

In [15]:
def check_level_up():

    while player.xp >= player.next_xp:

        player.xp -= player.next_xp

        player.level += 1

        player.max_hp += 5
        player.hp = player.max_hp

        player.attack += 1
        player.defense += 1

        player.next_xp = int(
            player.next_xp * 1.5
        )

        print("==============================")
        print("         LEVEL UP!")
        print("==============================")

        print(
            "현재 레벨:",
            player.level
        )

        print(
            "최대 HP:",
            player.max_hp
        )

        print(
            "공격력:",
            player.attack
        )

        print(
            "방어력:",
            player.defense
        )

        print("==============================")


# 테스트
gain_experience(20)
check_level_up()
show_player_status()

경험치 +20
현재 XP: 23/10
         LEVEL UP!
현재 레벨: 2
최대 HP: 25
공격력: 6
방어력: 3
        PLAYER STATUS
HP      : 25/25
ATK     : 6
DEF     : 3
LEVEL   : 2
XP      : 13/15
POSITION: (53, 3)


## 3-8 플레이어 사망

In [16]:
def check_player_death():

    if player.hp <= 0:

        player.hp = 0

        player.alive = False

        print()
        print("==============================")
        print("          YOU DIED")
        print("==============================")
        print("GAME OVER")
        print("==============================")

        return True

    return False


def damage_player(damage):

    player.hp -= damage

    print(
        f"플레이어가 {damage}의 피해를 입었습니다."
    )

    check_player_death()

# 제 4장 몬스터 시스템

## 4-1 몬스터 기본 클레스

In [17]:
class Monster:

    def __init__(
        self,
        name,
        char,
        x,
        y,
        hp,
        attack,
        defense,
        xp
    ):

        self.name = name
        self.char = char

        self.x = x
        self.y = y

        self.max_hp = hp
        self.hp = hp

        self.attack = attack
        self.defense = defense

        self.xp = xp

        self.alive = True


print("Monster 클래스 생성 완료")

Monster 클래스 생성 완료


## 4-2 첫번째 몬스터

In [18]:
rat = Monster(
    name="Rat",
    char="r",
    x=10,
    y=10,
    hp=5,
    attack=2,
    defense=0,
    xp=3
)

print("몬스터:", rat.name)
print("HP:", rat.hp)
print("공격력:", rat.attack)
print("방어력:", rat.defense)

몬스터: Rat
HP: 5
공격력: 2
방어력: 0


## 4-3 여러 종류의 몬스터

In [19]:
MONSTER_TYPES = {

    "Rat": {
        "char": "r",
        "hp": 5,
        "attack": 2,
        "defense": 0,
        "xp": 3
    },

    "Goblin": {
        "char": "g",
        "hp": 10,
        "attack": 4,
        "defense": 1,
        "xp": 5
    },

    "Orc": {
        "char": "o",
        "hp": 15,
        "attack": 6,
        "defense": 2,
        "xp": 8
    },

    "Troll": {
        "char": "T",
        "hp": 25,
        "attack": 8,
        "defense": 3,
        "xp": 12
    }
}


def create_monster(
    name,
    x,
    y
):

    data = MONSTER_TYPES[name]

    return Monster(
        name=name,
        char=data["char"],
        x=x,
        y=y,
        hp=data["hp"],
        attack=data["attack"],
        defense=data["defense"],
        xp=data["xp"]
    )


goblin = create_monster(
    "Goblin",
    15,
    10
)

print(
    goblin.name,
    goblin.hp,
    goblin.attack
)

Goblin 10 4


## 4-4 몬스터 랜덤 생성

In [20]:
monsters = []


def is_position_occupied(x, y):

    # 플레이어
    if (
        player.x == x
        and
        player.y == y
    ):
        return True

    # 다른 몬스터
    for monster in monsters:

        if (
            monster.x == x
            and
            monster.y == y
        ):
            return True

    return False


def get_random_floor_position():

    while True:

        x = random.randint(
            1,
            MAP_WIDTH - 2
        )

        y = random.randint(
            1,
            MAP_HEIGHT - 2
        )

        if dungeon[y][x] != FLOOR:
            continue

        if is_position_occupied(x, y):
            continue

        return x, y


def spawn_monster_random():

    name = random.choice(
        list(MONSTER_TYPES.keys())
    )

    x, y = get_random_floor_position()

    monster = create_monster(
        name,
        x,
        y
    )

    monsters.append(
        monster
    )


# 몬스터 5마리 생성
for _ in range(5):

    spawn_monster_random()


print(
    "생성된 몬스터:",
    len(monsters)
)

생성된 몬스터: 5


## 4-5 몬스터 이동

In [21]:
def move_monster(monster):

    directions = [
        (0, -1),
        (0, 1),
        (-1, 0),
        (1, 0)
    ]

    random.shuffle(
        directions
    )

    for dx, dy in directions:

        nx = monster.x + dx
        ny = monster.y + dy

        if nx < 0 or nx >= MAP_WIDTH:
            continue

        if ny < 0 or ny >= MAP_HEIGHT:
            continue

        if dungeon[ny][nx] != FLOOR:
            continue

        if (
            nx == player.x
            and
            ny == player.y
        ):
            continue

        occupied = False

        for other in monsters:

            if other is monster:
                continue

            if (
                other.x == nx
                and
                other.y == ny
            ):

                occupied = True
                break

        if occupied:
            continue

        monster.x = nx
        monster.y = ny

        break

## 4-6 플레이어 추적

In [22]:
def monster_distance(monster):

    return (
        abs(monster.x - player.x)
        +
        abs(monster.y - player.y)
    )


def chase_player(monster):

    distance = monster_distance(
        monster
    )

    # 너무 멀면 추적하지 않음
    if distance > 8:
        return

    dx = player.x - monster.x
    dy = player.y - monster.y

    if abs(dx) > abs(dy):

        step_x = 1 if dx > 0 else -1
        step_y = 0

    else:

        step_x = 0
        step_y = 1 if dy > 0 else -1

    nx = monster.x + step_x
    ny = monster.y + step_y

    if dungeon[ny][nx] != FLOOR:
        return

    if (
        nx == player.x
        and
        ny == player.y
    ):
        return

    monster.x = nx
    monster.y = ny

## 4-7 공격거리 판정

In [23]:
def is_adjacent_to_player(monster):

    distance = (
        abs(monster.x - player.x)
        +
        abs(monster.y - player.y)
    )

    return distance == 1


def monster_can_attack(monster):

    return is_adjacent_to_player(
        monster
    )

## 4-8 몬스터 사망

In [24]:
def damage_monster(
    monster,
    damage
):

    monster.hp -= damage

    print(
        f"{monster.name}이(가) "
        f"{damage}의 피해를 입었습니다."
    )

    if monster.hp <= 0:

        monster.hp = 0

        monster.alive = False

        print(
            f"{monster.name} 처치!"
        )

        gain_experience(
            monster.xp
        )

        check_level_up()

        if monster in monsters:

            monsters.remove(
                monster
            )

# 제 5장 전투 시스템

## 5-1 플레이어 공격

In [25]:
def get_monster_at(x, y):

    for monster in monsters:

        if (
            monster.x == x
            and
            monster.y == y
        ):

            return monster

    return None


def player_attack(dx, dy):

    target_x = player.x + dx
    target_y = player.y + dy

    monster = get_monster_at(
        target_x,
        target_y
    )

    if monster is None:

        print(
            "그 방향에는 공격할 몬스터가 없습니다."
        )

        return

    calculate_player_damage(
        monster
    )

## 5-2 기본 데미지 계산

In [26]:
def calculate_player_damage(
    monster
):

    damage = (
        player.attack
        -
        monster.defense
    )

    damage = max(
        1,
        damage
    )

    damage_monster(
        monster,
        damage
    )

## 5-3 랜덤 데미지

In [27]:
def calculate_player_damage_random(
    monster
):

    damage = random.randint(
        max(
            1,
            player.attack - 2
        ),
        player.attack + 2
    )

    damage -= monster.defense

    damage = max(
        1,
        damage
    )

    # 치명타
    if random.random() < 0.10:

        damage *= 2

        print("치명타!")

    damage_monster(
        monster,
        damage
    )

## 5-4 몬스터 반격

In [28]:
def monster_attack(monster):

    damage = random.randint(
        max(
            1,
            monster.attack - 1
        ),
        monster.attack + 1
    )

    damage -= player.defense

    damage = max(
        1,
        damage
    )

    print(
        f"{monster.name}이(가) "
        f"플레이어를 공격했습니다."
    )

    damage_player(
        damage
    )

## 5-5 전투 로그

In [29]:
combat_log = []


def add_combat_log(message):

    combat_log.append(
        message
    )

    if len(combat_log) > 8:

        combat_log.pop(0)


def show_combat_log():

    print()
    print("========== 전투 로그 ==========")

    for message in combat_log:

        print(
            ">",
            message
        )

    print("===============================")


# 기존 함수에 로그 기능 추가
def log_player_attack(
    monster,
    damage
):

    add_combat_log(
        f"{monster.name}에게 "
        f"{damage} 피해를 입혔습니다."
    )

## 5-6 전투 경험치 연결

In [30]:
def attack_monster(
    monster
):

    damage = random.randint(
        max(
            1,
            player.attack - 2
        ),
        player.attack + 2
    )

    damage -= monster.defense

    damage = max(
        1,
        damage
    )

    monster.hp -= damage

    message = (
        f"{monster.name}에게 "
        f"{damage} 피해!"
    )

    print(message)

    add_combat_log(
        message
    )

    # 몬스터 사망
    if monster.hp <= 0:

        monster.hp = 0
        monster.alive = False

        message = (
            f"{monster.name} 처치!"
        )

        print(message)

        add_combat_log(
            message
        )

        # 경험치
        player.xp += monster.xp

        message = (
            f"경험치 +{monster.xp}"
        )

        print(message)

        add_combat_log(
            message
        )

        # 레벨업
        check_level_up()

        if monster in monsters:

            monsters.remove(
                monster
            )

        return


    # 몬스터가 살아 있으면 반격
    monster_attack(
        monster
    )

## 5-7 플레이어 사망과 게임 오버

In [31]:
game_over = False


def check_game_over():

    global game_over

    if player.hp <= 0:

        player.hp = 0

        player.alive = False

        game_over = True

        print()
        print("================================")
        print("            YOU DIED")
        print("================================")
        print("GAME OVER")
        print("================================")

        return True

    return False

## 5-8 전투 시스템 통합

In [32]:
def player_turn(dx, dy):

    global turn

    if game_over:

        print("게임 오버 상태입니다.")

        return

    target_x = player.x + dx
    target_y = player.y + dy

    monster = get_monster_at(
        target_x,
        target_y
    )

    # 몬스터가 있으면 공격
    if monster:

        attack_monster(
            monster
        )

    else:

        # 없으면 이동
        move_player(
            dx,
            dy
        )

    # 턴 증가
    turn += 1

    # 살아있는 몬스터 행동
    if not game_over:

        for monster in list(monsters):

            if not monster.alive:
                continue

            if monster_can_attack(
                monster
            ):

                monster_attack(
                    monster
                )

            else:

                chase_player(
                    monster
                )

            if check_game_over():

                break

    print()
    print(
        f"현재 턴: {turn}"
    )

    show_player_status()

# 제 6장 아이템 시스템

## 6-1 item 클래스

In [33]:
class Item:
    def __init__(self, name, symbol, item_type, value=0):
        self.name = name
        self.symbol = symbol
        self.item_type = item_type
        self.value = value

## 6-2 아이템 종류 만들기

In [34]:
ITEM_TYPES = {
    "Potion": {
        "symbol": "!",
        "item_type": "heal",
        "value": 10
    },
    "Gold": {
        "symbol": "$",
        "item_type": "gold",
        "value": 10
    }
}

## 6-3 플레이어 인벤토리 추가

In [35]:
player.inventory = []
player.gold = 0

## 6-4 아이템 생성

In [36]:
def create_item(item_name):
    data = ITEM_TYPES[item_name]

    return Item(
        item_name,
        data["symbol"],
        data["item_type"],
        data["value"]
    )

## 6-5 아이템 위치 관리

In [37]:
items = []

def is_item_position(x, y):
    for item, ix, iy in items:
        if ix == x and iy == y:
            return True

    return False

## 6-6 아이템 랜덤 배치

In [38]:
def spawn_item_random(item_name):
    x, y = get_random_floor_position()

    while (
        (x == player.x and y == player.y)
        or is_position_occupied(x, y)
        or is_item_position(x, y)
    ):
        x, y = get_random_floor_position()

    item = create_item(item_name)
    items.append((item, x, y))

## 6-7 아이템 표시

In [39]:
def get_item_at(x, y):
    for item, ix, iy in items:
        if ix == x and iy == y:
            return item

    return None
def draw_item_map():
    for item, x, y in items:
        print(item.symbol, end="")

## 6-8 아이템 줍기

In [40]:
def pickup_item():
    for i, (item, x, y) in enumerate(items):
        if x == player.x and y == player.y:

            if item.item_type == "gold":
                player.gold += item.value
                print(f"{item.value} 골드를 얻었습니다.")

            else:
                player.inventory.append(item)
                print(f"{item.name}을(를) 얻었습니다.")

            items.pop(i)
            return

    print("여기에는 아이템이 없습니다.")

## 6-9 물약 사용

In [41]:
def use_item():
    if not player.inventory:
        print("인벤토리가 비어 있습니다.")
        return

    for i, item in enumerate(player.inventory):
        print(f"{i + 1}. {item.name}")

    try:
        choice = int(input("사용할 아이템 번호: ")) - 1
    except ValueError:
        print("잘못된 입력입니다.")
        return

    if choice < 0 or choice >= len(player.inventory):
        print("잘못된 번호입니다.")
        return

    item = player.inventory[choice]

    if item.item_type == "heal":
        old_hp = player.hp
        player.hp = min(player.max_hp, player.hp + item.value)

        healed = player.hp - old_hp

        print(f"{item.name}을(를) 사용했습니다.")
        print(f"HP +{healed}")

        player.inventory.pop(choice)

## 6-10 인벤토리 확인

In [42]:
def show_inventory():
    print("\n===== INVENTORY =====")

    if not player.inventory:
        print("비어 있음")
    else:
        for i, item in enumerate(player.inventory):
            print(f"{i + 1}. {item.name}")

    print(f"Gold: {player.gold}")
    print("=====================")

# 제 7장 던전 층 시스템

## 7-1 현재 층 추가

In [43]:
current_floor = 1
max_floor = 5

## 7-2 계단 클레스

In [44]:
class Stairs:
    def __init__(self, x, y):
        self.x = x
        self.y = y
        self.symbol = ">"

## 7-3 계단 생성

In [45]:
stairs = None

def create_stairs():
    global stairs

    x, y = get_random_floor_position()

    while (
        x == player.x and
        y == player.y
    ):
        x, y = get_random_floor_position()

    stairs = Stairs(x, y)

## 7-4 계단 위치 확인

In [46]:
def is_on_stairs():
    if stairs is None:
        return False

    return player.x == stairs.x and player.y == stairs.y

## 7-5 새로운 던전 생성

In [47]:
def generate_new_floor():
    global dungeon
    global rooms
    global monsters
    global items
    global traps
    global stairs
    global treasure

    dungeon, rooms = generate_dungeon()

    monsters = []
    items = []
    traps = []

    player.x, player.y = rooms[0].center()

    stairs = None
    treasure = None

    create_stairs()

    spawn_floor_monsters()

    for _ in range(current_floor):
        spawn_item_random("Potion")

    for _ in range(2):
        spawn_item_random("Gold")

    for _ in range(2):
        spawn_trap_random()

    # 마지막 층에만 보물 생성
    if current_floor == max_floor:
        create_treasure()

    reset_exploration()

## 7-6 층 이동

In [48]:
def go_to_next_floor():
    global current_floor

    if not is_on_stairs():
        print("계단 위에 있지 않습니다.")
        return

    if current_floor >= max_floor:
        print("더 이상 내려갈 수 없습니다.")
        return

    current_floor += 1

    print(f"{current_floor}층으로 내려갑니다.")

    generate_new_floor()

## 7-7 층 표시

In [49]:
def show_floor():
    print(f"현재 층: {current_floor} / {max_floor}")

## 7-8 난이도 증가

In [50]:
def get_monster_count():
    return 2 + current_floor * 2
def spawn_floor_monsters():
    for _ in range(get_monster_count()):
        spawn_monster_random()

# 제 8장 시야&탐험 시스템


## 8-1 탐험한 위치 저장

In [51]:
explored = set()

## 8-2 플레이어 주변 탐색

In [52]:
def reveal_area(radius=5):
    for y in range(
        max(0, player.y - radius),
        min(MAP_HEIGHT, player.y + radius + 1)
    ):
        for x in range(
            max(0, player.x - radius),
            min(MAP_WIDTH, player.x + radius + 1)
        ):
            distance = abs(player.x - x) + abs(player.y - y)

            if distance <= radius:
                explored.add((x, y))

## 8-3 탐험 여부 확인

In [53]:
def is_explored(x, y):
    return (x, y) in explored

## 8-4 새로운 위치 발견

In [54]:
def explore_current_position():
    reveal_area(5)

## 8-5 시야가 없는 곳 표시

In [55]:
def draw_fog_map():
    for y in range(MAP_HEIGHT):
        line = ""

        for x in range(MAP_WIDTH):

            if not is_explored(x, y):
                line += " "
                continue

            if x == player.x and y == player.y:
                line += "@"
                continue

            monster = get_monster_at(x, y)

            if monster:
                line += monster.symbol
                continue

            item = get_item_at(x, y)

            if item:
                line += item.symbol
                continue

            if stairs and stairs.x == x and stairs.y == y:
                line += stairs.symbol
                continue

            line += dungeon[y][x]

        print(line)

## 8-6 층 변경 시 탐험 초기화




In [56]:
def reset_exploration():
    global explored

    explored = set()
    reveal_area(5)

## 8-7 새로운 층 생성에 시야 적용

In [57]:
def generate_new_floor_with_fog():
    global dungeon
    global rooms
    global monsters
    global items
    global stairs

    dungeon, rooms = generate_dungeon()

    monsters = []
    items = []

    player.x, player.y = rooms[0].center()

    create_stairs()

    spawn_floor_monsters()

    for _ in range(current_floor):
        spawn_item_random("Potion")

    reset_exploration()

# 제 9장 함정&상태 시스템

## 9-1 Trap 클래스

In [58]:
class Trap:
    def __init__(self, x, y, damage):
        self.x = x
        self.y = y
        self.damage = damage
        self.symbol = "^"
        self.visible = False

## 9-2 함정 목록

In [59]:
traps = []

## 9-3 함정 생성

In [60]:
def spawn_trap_random():
    x, y = get_random_floor_position()

    while (
        (x == player.x and y == player.y)
        or is_position_occupied(x, y)
        or is_item_position(x, y)
    ):
        x, y = get_random_floor_position()

    trap = Trap(
        x,
        y,
        random.randint(2, 5) + current_floor
    )

    traps.append(trap)

## 9-4 함정 확인

In [61]:
def get_trap_at(x, y):
    for trap in traps:
        if trap.x == x and trap.y == y:
            return trap

    return None

## 9-5 함정 발동

In [62]:
def check_trap():
    trap = get_trap_at(player.x, player.y)

    if trap is None:
        return

    trap.visible = True

    print("함정을 밟았습니다!")

    damage_player(trap.damage)

    traps.remove(trap)

## 9-6 함정 표시

In [63]:
def draw_traps():
    for trap in traps:
        if trap.visible:
            print(f"함정 위치: ({trap.x}, {trap.y})")

## 9-7 이동 후 함정 검사

In [64]:
def move_player_and_check_trap(dx, dy):
    old_x = player.x
    old_y = player.y

    move_player(dx, dy)

    if player.x != old_x or player.y != old_y:
        check_trap()

## 9-8 독 상태 추가

In [65]:
player.poisoned = False
player.poison_turns = 0

## 9-9 독 데미지

In [66]:
def apply_poison():
    if not player.poisoned:
        return

    damage_player(1)

    player.poison_turns -= 1

    print("독 때문에 HP가 1 감소했습니다.")

    if player.poison_turns <= 0:
        player.poisoned = False
        print("독이 사라졌습니다.")

## 9-10 독 상태 적용

In [67]:
def poison_player(turns=5):
    player.poisoned = True
    player.poison_turns = turns

    print("독에 걸렸습니다!")

# 제 10장 최종 목표& 게임 완성

## 10-1 최종 아이템

In [68]:
class Treasure:
    def __init__(self, x, y):
        self.x = x
        self.y = y
        self.symbol = "*"

## 10-2 보물 생성

In [69]:
treasure = None

def create_treasure():
    global treasure

    x, y = get_random_floor_position()

    while (
        x == player.x and
        y == player.y
    ):
        x, y = get_random_floor_position()

    treasure = Treasure(x, y)

## 10-3 보물 확인

In [70]:
def is_on_treasure():
    if treasure is None:
        return False

    return (
        player.x == treasure.x and
        player.y == treasure.y
    )

## 10-4 최종 보물 획득

In [71]:
game_won = False

def collect_treasure():
    global game_won

    if not is_on_treasure():
        return

    if current_floor < max_floor:
        print("아직 최종 층이 아닙니다.")
        return

    print()
    print("================================")
    print("        TREASURE FOUND!")
    print("        던전을 정복했습니다!")
    print("================================")

    game_won = True

## 10-5 게임 종료 확인

In [72]:
def is_game_over():
    return not player.alive or game_won

## 10-6 게임 상태 표시

In [73]:
def show_game_status():
    print()
    print("========== GAME STATUS ==========")
    print(f"Floor : {current_floor}")
    print(f"HP    : {player.hp}/{player.max_hp}")
    print(f"Level : {player.level}")
    print(f"XP    : {player.xp}/{player.next_xp}")
    print(f"Gold  : {player.gold}")
    print("=================================")

## 10-7 명령어 처리

In [74]:
def process_command(command):

    # None이나 문자열이 아닌 입력 방지
    if command is None:
        return False

    command = str(command).lower().strip()

    if command == "w":
        move_player_and_check_trap(0, -1)
        return True

    elif command == "s":
        move_player_and_check_trap(0, 1)
        return True

    elif command == "a":
        move_player_and_check_trap(-1, 0)
        return True

    elif command == "d":
        move_player_and_check_trap(1, 0)
        return True

    elif command == "g":
        pickup_item()
        collect_treasure()
        return True

    elif command == "i":
        show_inventory()
        return False

    elif command == "u":
        use_item()
        return True

    elif command == ">":
        go_to_next_floor()
        return True

    elif command == "c":
        show_game_status()
        return False

    elif command == "q":
        return None

    else:
        return False

## 10-8 몬스터 턴


In [75]:
def player_turn(dx, dy):

    global turn
    global game_over

    if game_over or not player.alive:
        return False

    target_x = player.x + dx
    target_y = player.y + dy

    # 몬스터가 있으면 공격
    monster = get_monster_at(
        target_x,
        target_y
    )

    if monster:

        attack_monster(monster)

        turn += 1

    else:

        old_x = player.x
        old_y = player.y

        move_player(dx, dy)

        # 실제로 이동했을 때만 턴 증가
        if (
            player.x != old_x
            or
            player.y != old_y
        ):

            check_trap()

            turn += 1

        else:

            return False

    # 독
    if player.alive:
        apply_poison()

    # 몬스터 행동
    if player.alive:
        monsters_turn()

    # 시야 갱신
    if player.alive:
        reveal_area(5)

    check_game_over()

    return True

## 10-9 플레이어 턴 통합

In [76]:
def full_player_turn(command):

    result = process_command(command)

    if result is None:
        return False

    if result:
        apply_poison()

        if not player.alive:
            return False

        monsters_turn()

        if not player.alive:
            return False

        reveal_area(5)

    return True

## 10-10 최종 게임 루프

In [77]:
import ipywidgets as widgets
from IPython.display import display, clear_output

game_output = widgets.Output()

game_running = False

10-10-1 게임 초기화

In [78]:
import html
import base64
from IPython.display import display, HTML, Javascript
from google.colab import output

game_running = False
game_won = False


def initialize_game():

    global dungeon
    global rooms
    global monsters
    global items
    global traps
    global stairs
    global treasure
    global current_floor
    global game_running
    global game_won
    global explored

    current_floor = 1
    game_running = True
    game_won = False

    # 플레이어 초기화
    player.hp = player.max_hp
    player.alive = True

    player.xp = 0
    player.level = 1
    player.next_xp = 10

    player.gold = 0
    player.inventory = []

    player.poisoned = False
    player.poison_turns = 0

    # 던전 생성
    dungeon, rooms = generate_dungeon()

    # 기존 객체 초기화
    monsters = []
    items = []
    traps = []

    stairs = None
    treasure = None

    explored = set()

    # 첫 번째 방 중앙에서 시작
    player.x, player.y = rooms[0].center()

    # 계단 생성
    create_stairs()

    # 층 몬스터 생성
    spawn_floor_monsters()

    # 포션
    for _ in range(3):
        spawn_item_random("Potion")

    # 골드
    for _ in range(2):
        spawn_item_random("Gold")

    # 함정
    for _ in range(2):
        spawn_trap_random()

    # 마지막 층이면 보물 생성
    if current_floor == max_floor:
        create_treasure()

    # 주변 시야 공개
    reveal_area(5)

10-10-2 검은색 게임 화

In [79]:
def get_game_screen():

    lines = []

    lines.append("=" * 60)
    lines.append("                 ASCII ROGUE")
    lines.append(
        f"                 FLOOR {current_floor}/{max_floor}"
    )
    lines.append("=" * 60)

    for y in range(MAP_HEIGHT):

        line = ""

        for x in range(MAP_WIDTH):

            # 아직 탐색하지 않은 곳
            if not is_explored(x, y):
                line += " "
                continue

            # 플레이어
            if x == player.x and y == player.y:
                line += "@"
                continue

            # 몬스터
            monster = get_monster_at(x, y)

            if monster and monster.alive:
                line += monster.char
                continue

            # 아이템
            item = get_item_at(x, y)

            if item:
                line += item.symbol
                continue

            # 계단
            if (
                stairs
                and x == stairs.x
                and y == stairs.y
            ):
                line += stairs.symbol
                continue

            # 보물
            if (
                treasure
                and x == treasure.x
                and y == treasure.y
            ):
                line += treasure.symbol
                continue

            # 기본 던전
            line += dungeon[y][x]

        lines.append(line)

    lines.append("")
    lines.append("=" * 60)

    lines.append(
        f"HP: {player.hp}/{player.max_hp}   "
        f"ATK: {player.attack}   "
        f"DEF: {player.defense}"
    )

    lines.append(
        f"LV: {player.level}   "
        f"XP: {player.xp}/{player.next_xp}   "
        f"GOLD: {player.gold}"
    )

    if player.poisoned:
        lines.append(
            f"상태: POISON ({player.poison_turns}턴)"
        )

    if is_on_stairs():
        lines.append("[>] 계단 위")

    if is_on_treasure():
        lines.append("[G] 최종 보물")

    lines.append("")
    lines.append("W A S D : 이동")
    lines.append("G : 아이템 줍기")
    lines.append("I : 인벤토리")
    lines.append("U : 아이템 사용")
    lines.append("C : 상태 확인")
    lines.append("> : 다음 층")
    lines.append("Q : 게임 종료")

    return "\n".join(lines)

10-10-3 키 입력 처리

In [80]:
def handle_key(key):

    global game_running

    if key is None:
        return get_game_screen()

    key = str(key).lower().strip()

    if not game_running:
        return get_game_screen()

    result = process_command(key)

    # 게임 종료
    if result is None:

        game_running = False

        return (
            get_game_screen()
            + "\n\n"
            + "============================\n"
            + "          GAME QUIT\n"
            + "============================"
        )

    # 턴을 소비하는 행동
    if result:

        # 독 데미지
        apply_poison()

        if not player.alive:

            game_running = False

            return (
                get_game_screen()
                + "\n\n"
                + "============================\n"
                + "          GAME OVER\n"
                + "============================"
            )

        # 몬스터 행동
        monsters_turn()

        if not player.alive:

            game_running = False

            return (
                get_game_screen()
                + "\n\n"
                + "============================\n"
                + "          GAME OVER\n"
                + "============================"
            )

        # 시야 갱신
        reveal_area(5)

    # 게임 클리어
    if game_won:

        game_running = False

        return (
            get_game_screen()
            + "\n\n"
            + "============================\n"
            + "         GAME CLEAR!\n"
            + "============================"
        )

    return get_game_screen()

10-10-4 파이썬<->JavaScript 연결

In [81]:
from google.colab import output
from IPython.display import display, HTML
import ipywidgets as widgets
import html


# 게임 화면 위젯
rogue_screen = widgets.HTML(
    value="",
    layout=widgets.Layout(
        width="800px",
        height="700px"
    )
)


# 화면을 HTML로 변환
def make_rogue_screen():

    screen = get_game_screen()

    return f"""
    <pre style="
        margin: 0;
        width: 760px;
        height: 660px;
        padding: 18px;
        box-sizing: border-box;

        background: black;
        color: white;

        border: 2px solid #555;
        border-radius: 8px;

        font-family: monospace;
        font-size: 14px;
        line-height: 1.15;

        white-space: pre;
        overflow: hidden;
    ">{html.escape(screen)}</pre>
    """


# 키 입력 처리
def rogue_key_callback(key):

    # 게임 명령 실행
    handle_key(key)

    # Python에서 화면 자체를 갱신
    rogue_screen.value = make_rogue_screen()

    return "OK"


# callback 등록
output.register_callback(
    "rogue.key",
    rogue_key_callback
)

In [82]:
from IPython.display import display, Javascript


# 게임 시작
initialize_game()


# 최초 화면
rogue_screen.value = make_rogue_screen()


# 화면 표시
display(rogue_screen)


# 키보드 연결
display(Javascript("""
(() => {

    // 이전 이벤트 제거
    if (window.rogueKeyboardHandler) {

        document.removeEventListener(
            "keydown",
            window.rogueKeyboardHandler,
            true
        );

    }


    // 사용할 키
    const validKeys = new Set([
        "w",
        "a",
        "s",
        "d",
        "g",
        "i",
        "u",
        "c",
        ">",
        "q"
    ]);


    // 키보드 이벤트
    window.rogueKeyboardHandler = function(event) {

        let key = event.key.toLowerCase();


        // 게임 키가 아니면 무시
        if (!validKeys.has(key)) {
            return;
        }


        // 브라우저 기본 동작 차단
        event.preventDefault();
        event.stopPropagation();


        // Python callback 실행
        google.colab.kernel.invokeFunction(
            "rogue.key",
            [key],
            {}
        );

    };


    // 키보드 이벤트 등록
    document.addEventListener(
        "keydown",
        window.rogueKeyboardHandler,
        true
    );


    console.log(
        "ROGUE KEYBOARD READY"
    );

})();
"""))

HTML(value='\n    <pre style="\n        margin: 0;\n        width: 760px;\n        height: 660px;\n        pad…

<IPython.core.display.Javascript object>

현재 위치: 52 17
현재 위치: 53 17
